In [24]:
import cv2
import torch
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import time

In [17]:
model = YOLO("yolov8s.pt")


00%|██████████| 21.5M/21.5M [00:00<00:00, 38.3MB/s]

In [31]:
model.names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [18]:
rtsp_url = "rtsp://<USER>:<PASSWORD>@<CAMERA_IP>:1554/stream1"

In [52]:
cap = cv2.VideoCapture(rtsp_url)
output_video_path = "tracking_video.mp4"
ret, frame = cap.read()

track_history = defaultdict(lambda: [])
if not (ret):
    print('exit')
    cap.release()
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
duration = 30
total_frames = fps * duration 
height, width = min(frame.shape[:2], frame.shape[:2])
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
start_time = time.time()
classes = model.names
while int(time.time() - start_time) < duration:
    success, frame = cap.read()

    if success:
        results = model.track(frame, persist=True)

        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        annotated_frame = results[0].plot()

        for box, track_id, class_id in zip(boxes, track_ids, class_ids):
            if class_id == 0:
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))
                if len(track) > 30:
                    track.pop(0)
    
                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.putText(frame, f"{classes[class_id]} : {str(track_id)}", (int(x), int(y) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                box_x = x - (w / 2)
                box_y = y - (h  / 2)
                cv2.rectangle(frame, (int(box_x), int(box_y), int(w), int(h)), (0, 255, 0), 2)
                cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)
                # cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

        out.write(frame)
    else:
        print('bad')
        break
print('                     end ')
cap.release()
out.release()


0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.8ms
Speed: 22.3ms preprocess, 6.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.4ms
Speed: 2.2ms preprocess, 6.4ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.2ms
Speed: 2.2ms preprocess, 6.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.3ms
Speed: 2.3ms preprocess, 6.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.1ms
Speed: 2.3ms preprocess, 6.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 suitcase, 10 chairs, 11 tvs, 4 keyboards, 1 clock, 6.1ms
Speed: 2.2ms preprocess, 6.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 64

In [29]:
track_history

defaultdict(<function __main__.<lambda>()>,
            {380: [(231.5674285888672, 246.97802734375),
              (231.60751342773438, 247.02159118652344),
              (231.62301635742188, 247.03802490234375),
              (231.6321258544922, 247.0446319580078),
              (231.63479614257812, 247.0465545654297),
              (231.63668823242188, 247.048828125),
              (231.6368865966797, 247.04888916015625),
              (231.6374969482422, 247.04812622070312),
              (231.63882446289062, 247.04856872558594),
              (231.64141845703125, 247.0470428466797),
              (231.63999938964844, 247.0513916015625),
              (231.64013671875, 247.05508422851562),
              (231.63931274414062, 247.05616760253906),
              (231.64028930664062, 247.056640625),
              (231.64016723632812, 247.05545043945312),
              (231.6398468017578, 247.05540466308594),
              (231.56686401367188, 247.34033203125),
              (231.52938842